# CDC PLACES Data Collection

This notebook pulls data from the CDC PLACES (Population Level Analysis and Community Estimates) API using the `CDCPLACES` R package.

## About CDC PLACES

PLACES provides model-based, population-level analysis and community estimates of health measures to all counties, places (incorporated and census designated places), census tracts, and ZIP Code Tabulation Areas (ZCTAs) across the United States.

**Source:** [CDCPLACES Package Documentation](https://brendensm.github.io/CDCPLACES/)

In [ ]:
# Install and load required packages
# Uncomment the install line if you need to install the package
# install.packages("CDCPLACES")
# Or install from GitHub: devtools::install_github("brendensm/CDCPLACES")
library(CDCPLACES)
library(dplyr)
library(ggplot2)

## Step 1: Explore Available Measures

First, let's get a dictionary of all available measures in the PLACES dataset.

In [ ]:
# Get dictionary of available measures
measures_dict <- get_dictionary()

# Display first few rows
head(measures_dict, 40)

# View structure
str(measures_dict)

# Count total measures
cat("Total number of available measures:", nrow(measures_dict), "\n")

## Step 2: Query PLACES Data

The `get_places()` function allows you to query data by:
- **Geography type**: county, census tract, ZCTA (ZIP Code), or place
- **State**: State abbreviation (e.g., "CA", "NY")
- **Measure**: Specific health measure ID from the dictionary
- **Release**: Data release year
- **Geometry**: Whether to include shapefiles (TRUE/FALSE)

### Example 1: Get county-level data for a specific state

In [ ]:
# Example: Get county-level data for California
# Replace "CA" with your desired state abbreviation
# Replace "MeasureID" with an actual measure ID from the dictionary

# Example: Get county-level data for California
# Replace "CA" and measure_id below with your desired state and measure.

# --- Step 1: Find relevant measures ---
# PLACES does not include pesticide-specific measures. For pesticide exposure
# research, common proxies are respiratory outcomes (e.g., asthma, COPD).
# Filter by searching in measure names (columns: measure_full_name, measure_short_name):
relevant_measures <- measures_dict %>%
  filter(
    grepl("asthma|COPD|respiratory|disease|health|prevalence",
          measure_full_name, ignore.case = TRUE) |
    grepl("asthma|COPD|respiratory|disease|health",
          measure_short_name, ignore.case = TRUE)
  )

# Show matching measures and their IDs (use measureid when calling get_places)
print("Example measures relevant to environmental / respiratory health:")
relevant_measures %>%
  select(measureid, measure_short_name, measure_full_name) %>%
  head(20)

# --- Step 2: Fetch county-level data for one measure and state ---
state_abbr <- "CA"        # California; use any state abbreviation
measure_id <- "CASTHMA"   # Current asthma among adults (from dictionary)

ca_county_data <- get_places(
  geography = "county",
  state = state_abbr,
  measure = measure_id,
  release = 2023,
  geometry = FALSE
)

# --- Step 3: Inspect the result ---
cat("\nCounty-level data for", measure_id, "in", state_abbr, ":\n")
cat("Rows:", nrow(ca_county_data), "\n")
head(ca_county_data)

### Example 2: Query specific measures for counties

In [ ]:
# Example query structure (uncomment and modify as needed)
# Replace STATE_ABBR with your state (e.g., "CA", "NY", "TX")
# Replace MEASURE_ID with a measure ID from the dictionary above

places_data <- get_places(
  geography = "county",
  state = "OH",  # e.g., "CA" for California
  measure = "COPD",  # Replace with actual measure ID
  release = 2023,  # Check available releases
  geometry = FALSE  # Set to TRUE if you need shapefiles
)
# View the data
head(places_data)
str(places_data)

### Example 3: Query multiple measures or geographies

You can query multiple measures by calling `get_places()` multiple times and combining the results.

In [ ]:
# Example: Query multiple measures for the same geography
# This is a template - customize based on your needs

measure_ids <- c("COPD", "CASTHMA")

all_data <- lapply(measure_ids, function(measure_id) {
  get_places(
    geography = "county",
    state = "OH",
    measure = measure_id,
    release = 2023,
    geometry = FALSE
  )
})

# Combine results
combined_data <- bind_rows(all_data)
head(combined_data)

## Step 3: Data Exploration and Analysis

Once you have the data, you can explore and analyze it.

In [ ]:
# Example data exploration (uncomment when you have data)
# 
# # Summary statistics
# summary(places_data)
# 
# # Check for missing values
# colSums(is.na(places_data))
# 
# # Basic visualization
# # ggplot(places_data, aes(x = Data_Value)) +
# #   geom_histogram(bins = 30) +
# #   labs(title = "Distribution of Health Measure Values",
# #        x = "Data Value",
# #        y = "Frequency")

## Next Steps

1. **Identify relevant measures**: Review the measures dictionary to find health measures related to your research question
2. **Select geography**: Choose the appropriate geography level (county, census tract, ZCTA, or place)
3. **Query data**: Use `get_places()` to pull the data for your selected measures and geographies
4. **Clean and analyze**: Process the data as needed for your analysis

## Resources

- [CDCPLACES Package Documentation](https://brendensm.github.io/CDCPLACES/)
- [CDC PLACES Website](https://www.cdc.gov/places/)
- [Package Vignette](https://brendensm.github.io/CDCPLACES/articles/CDCPLACES.html)